# Exercise 3

## 3.1) and 3.2)

In [79]:
from keras.datasets import imdb
from keras import models
from keras import layers
import numpy as np
import re

(imdbTrainReview, imdbTrainLabel), (imdbTestReview, imdbTestLabel) = imdb.load_data(num_words=10000)

data = np.concatenate((imdbTrainReview, imdbTestReview), axis=0)

wordIndex = imdb.get_word_index()
reverseIndex = dict([(i, w) for (w, i) in wordIndex.items()]) 

def decodeSequence(seq):
    return " ".join([reverseIndex.get(i - 3, "#") for i in seq])

def vectorize(seqs, dimension=10000):
    results = np.zeros((len(seqs), dimension))
    for i, sequence in enumerate(seqs):
        results[i, sequence] = 1
    return results

data = vectorize(data)

train_x = vectorize(imdbTrainReview)
train_y = imdbTrainLabel
test_x = vectorize(imdbTestReview)
test_y = imdbTestLabel

model = models.Sequential()

model.add(layers.Dense(50, activation="relu", input_shape=(10000, )))
model.add(layers.Dropout(0.4, noise_shape=None, seed=None))
model.add(layers.Dense(50, activation = "relu"))
model.add(layers.Dense(1, activation = "sigmoid"))

model.compile(
    optimizer="adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

results = model.fit(
 train_x, train_y,
 epochs= 2,
 batch_size = 500,
 validation_data = (test_x, test_y)
)

Epoch 1/2
50/50 [==============================] - 2s 35ms/step - loss: 0.4650 - accuracy: 0.7909 - val_loss: 0.2950 - val_accuracy: 0.8834
Epoch 2/2
50/50 [==============================] - 1s 19ms/step - loss: 0.2375 - accuracy: 0.9092 - val_loss: 0.2762 - val_accuracy: 0.8890


In [2]:
print("Test-Accuracy:", np.mean(results.history["val_accuracy"]))

Test-Accuracy: 0.8941666483879089


OUTPUT
```
Test-Accuracy: 0.8907999992370605
```

## 3.3)

In [3]:
from keras.layers import Embedding, Dropout, LSTM, Dense

max_features = 10000

def buildLSTM():
    model = models.Sequential()
    model.add(Embedding(max_features, 128, input_length=10000))
    model.add(Dropout(0.4,  noise_shape=None, seed=None))
    model.add(LSTM(200, dropout=0.2, recurrent_dropout=0.2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
    return model

LSTMmodel = buildLSTM()

print(train_x.shape)
print(train_y.shape)
print(test_x.shape)
print(test_y.shape)

(25000, 10000)
(25000,)
(25000, 10000)
(25000,)


In [6]:
LSTMresults = LSTMmodel.fit(
 train_x[:1000], train_y[:1000],
 epochs=1,
 batch_size=16,
 validation_data = (test_x[1000:1500], test_y[1000:1500])
)

print("Test-Accuracy:", np.mean(LSTMresults.history["val_accuracy"]))

63/63 [==============================] - 1363s 22s/step - loss: 0.6954 - accuracy: 0.4850 - val_loss: 0.6896 - val_accuracy: 0.5560
Test-Accuracy: 0.5559999942779541


_Note_: In this exercise I could have formatted the textual data in an other way to make this faster, such as a key phrase extraction as they did in the book. But since I do not have much time I just reduced the data (which is not good practice:))), and will in the project use better information extraction methods!

OUTPUT
```
63/63 [==============================] - 1363s 22s/step - loss: 0.6954 - accuracy: 0.4850 - val_loss: 0.6896 - val_accuracy: 0.5560
Test-Accuracy: 0.5559999942779541
```

A bit better than guessing:D

## 3.5)

In [81]:
def encodeSentence(sent):
    sent = re.sub('\W+', ' ', sent)
    return np.array([wordIndex.get(word.lower(), "") for word in sent.split(" ") if not wordIndex.get(word.lower(), "#") == "#" and wordIndex.get(word.lower()) < 10000])

def encodeSentences(sents):
    return [encodeSentence(sent) for sent in sents]

def prepareTextSents(sents):
    return vectorize(encodeSentences(sents))

def predict(model, inputs):
    return model.predict(inputs)


harryPotterSents = ["hate", "Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly normal, thank you very much. They were the last people you'd expect to be involved in anything strange or mysterious, because they just didn't hold with such nonsense.",
"The Potters, that's right, that's what I heard yes, their son, Harry",
"At school, Harry had no one. Everybody knew that Dudley's gang hated that odd Harry Potter in his baggy old clothes and broken glasses, and nobody liked to disagree with Dudley's gang.",
"Harry tried to wave the owl out of the way, but it snapped its beak fiercely at him and carried on savaging the coat."]

harryPotterEncoded = prepareTextSents(harryPotterSents)

predict(LSTMmodel, harryPotterEncoded)

array([[0.48098278],
       [0.48098278],
       [0.48098278],
       [0.48098278],
       [0.48098278]], dtype=float32)

OUTPUT
```
array([[0.48098278],
       [0.48098278],
       [0.48098278],
       [0.48098278],
       [0.48098278]], dtype=float32)
```
This shows some flaws with my model, as all the sentences give the same output. It is probably the lack of test data and small number of epochs that lead to this. If I had more time I would fix this. I will definetly take this into consideration in the project. To create a better dataset for fantasy I would probably use books as my raw dataset to get words that are "fantasy-like", since fantasy writers can get quite creative in how they formulate things.